#### **DPR**

In [1]:
%load_ext autoreload
%autoreload 2

# from src.retrievers.models.encoder import TransformerEncoder
from src.retrievers.dpr import DPR

from hydra import initialize, compose

with initialize(config_path="config", version_base=None):
    cfg = compose(config_name="config")

dpr = DPR(cfg)

/Users/rasmuslinnemann/Desktop/Courses/Semester7/02456 Deep Learning/Project/02456-retrieval-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1121 14:14:32.987000 48471 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11772.45it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls

In [2]:
print(dpr.query_encoder.model.config)               # BERT's full config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.22.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [5]:
input_query = dpr.query_encoder.tokenizer("dogs are cute", return_tensors="pt")

input_doc = dpr.doc_encoder.tokenizer("dogs are cute", return_tensors="pt")

In [ ]:
output_query = dpr.query_encoder.model(**input_query)

output_doc = dpr.doc_encoder.model(**input_doc)

print(output_query.last_hidden_state.shape)
print(output_doc.last_hidden_state.shape)

torch.Size([1, 5, 768])
torch.Size([1, 5, 768])


In [ ]:
output_query_pooled = output_query.pooler_output
output_doc_pooled = output_doc.pooler_output

print(output_query_pooled.shape)
print(output_doc_pooled.shape)

torch.Size([1, 768])
torch.Size([1, 768])


In [ ]:
output_query_pooled @ output_doc_pooled.T

tensor([[224.3565]], grad_fn=<MmBackward0>)

In [ ]:
dpr.query_document(["dogs are cute","cats are cool"], ["dogs are cute","cats are cool"])

tensor([[224.3566, 213.0895],
        [213.0895, 227.7631]], grad_fn=<MmBackward0>)

In [ ]:
print(dpr.query_encoder.model.config.hidden_size)   # hidden dim

768


In [ ]:
print(dpr.query_encoder.model)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [ ]:
# BertForPreTraining  =  body  +  MLM head  +  NSP head
# BertModel           =  body


In [ ]:
# query    → query BERT → pool → q  (768)
# document → doc BERT   → pool → d  (768)
# score = q · d          ← dot product; high means relevant


In [2]:
from src.data_utils.corpus import Corpus
from src.data_utils.query import Query

corpus = Corpus(dataset_name=cfg.dataset.name, base_dir=f'{cfg.base_path}/retrieval_data')

### Prepare queries
train_queries = [Query(corpus.train['question'][idx], 
                        id=corpus.train['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.train['query_id'][idx]]
                        ) 
                    for idx in range(corpus.train.num_rows)
                ]

val_queries =   [Query(corpus.val['question'][idx], 
                        id=corpus.val['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.val['query_id'][idx]]
                        ) 
                    for idx in range(corpus.val.num_rows)
                ]

test_queries =  [Query(corpus.test['question'][idx], 
                        id=corpus.test['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.test['query_id'][idx]]
                        ) 
                    for idx in range(corpus.test.num_rows)
                ]

In [3]:
print(train_queries[5].text)
print(train_queries[5].relevant_document_ids)

Hobby vs. Business
['doc7', 'doc8']


In [4]:
input_query = dpr.query_encoder.tokenizer(train_queries[5].text, return_tensors="pt")

input_doc = dpr.doc_encoder.tokenizer("dogs are cute", return_tensors="pt")
output_query = dpr.query_encoder.model(**input_query)

output_doc = dpr.doc_encoder.model(**input_doc)

print(output_query.last_hidden_state.shape)
print(output_doc.last_hidden_state.shape)

output_query_pooled = output_query.pooler_output
output_doc_pooled = output_doc.pooler_output

print(output_query_pooled.shape)
print(output_doc_pooled.shape)

# dpr.query_document(["dogs are cute","cats are cool"], ["dogs are cute","cats are cool"])

torch.Size([1, 6, 768])
torch.Size([1, 5, 768])
torch.Size([1, 768])
torch.Size([1, 768])


In [ ]:
from src.training.loss import DPRLoss

loss = DPRLoss()

loss.forward(output_query_pooled, output_doc_pooled) # score (temp)

tensor([[199.4550]], grad_fn=<MmBackward0>)

In [ ]:

criterion = DPRLoss()
q = dpr.encode_queries(query_texts)
d = dpr.encode_documents(doc_texts)
loss = criterion(q, d)


In [ ]:
for queries, documents in dataloader:          # lists of strings
    scores = dpr.query_document(queries, documents)   # model forward → (B, B)
    loss = criterion(scores)                           # loss forward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


In [30]:
corpus.train

Dataset({
    features: ['question', 'text', 'query_id', 'document_id'],
    num_rows: 5500
})

In [43]:
train_queries[5].text

'Hobby vs. Business'

In [42]:
train_queries[5].relevant_document_ids

['doc7', 'doc8']

In [3]:
from src.training.loss import DPRLoss

loss = DPRLoss()

loss.forward([],[])

NotImplementedError: 

In [4]:
output_query

NameError: name 'output_query' is not defined

In [7]:
from datasets import load_from_disk

# Load the entire dataset directory
ds = load_from_disk("retrieval_data/fiqa")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 5500
    })
    validation: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 500
    })
    test: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 648
    })
})


In [18]:
ds["train"]

Dataset({
    features: ['question', 'text', 'query_id', 'document_id'],
    num_rows: 5500
})

In [23]:
ds["train"][5]

{'question': 'Hobby vs. Business',
 'text': ['Miscellaneous income -- same category used for hobbies.',
  '"You can list it as other income reported on line 21 of form 1040.  In TurboTax, enter at: - Federal Taxes tab (Personal in  Home & Business) - Wages & Income -“I’ll choose what I work on” Button Scroll down to: -Less Common Income -Misc Income, 1099-A, 1099-C. -The next screen will give you several choices. Choose ""Other reportable Income"". You will reach a screen where you can type a description of the income and the amount. Type in the amount of income and categorize as Tutoring."'],
 'query_id': 'q5',
 'document_id': ['doc7', 'doc8']}

In [14]:
ds["train"][0]["question"]

'What is considered a business expense on a business trip?'

In [17]:
query = ds["train"][0]["question"]
document = ds["train"][0]["text"]

wrong_document = ["asdh kajshd hasd haish dashdjkh ajsd ashdasd jasdh ahsjd hasjdh ajshdjashdjas hdjhas hasdh kajshd hasd haish dashdjkh ajsd ashdasd jasdh ahsjd hasjdh ajshdjashdjas hdjhas hasdh kajshd hasd haish dashdjkh ajsd ashdasd jasdh ahsjd hasjdh ajshdjashdjas hdjhas hasdh kajshd hasd haish dashdjkh ajsd ashdasd jasdh ahsjd hasjdh ajshdjashdjas hdjhas hasdh kajshd hasd haish dashdjkh ajsd ashdasd jasdh ahsjd hasjdh ajshdjashdjas hdjhas hasdh kajshd hasd haish dashdjkh ajsd ashdasd jasdh ahsjd hasjdh ajshdjashdjas hdjhas hasdh kajshd hasd haish dashdjkh ajsd ashdasd jasdh ahsjd hasjdh ajshdjashdjas hdjhas hasdh kajshd hasd haish dashdjkh ajsd ashdasd jasdh ahsjd hasjdh ajshdjashdjas hdjhas h"]

dpr.query_document(query, document)

tensor([[244.5191]], grad_fn=<MmBackward0>)